# 03 · Kvasir-SEG: EIoU vs AEIoU Thorough Comparison

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nipun-taneja/amorphous-yolo/blob/main/notebooks/03_kvasir_eiou_vs_aeiou.ipynb)

## Abstract

This notebook is a focused companion to `02_full_loss_comparison.ipynb`.
Where notebook 02 benchmarks **all 7 IoU-based losses** on the DUO underwater dataset,
this notebook zooms in on a single research question:

> **Does AEIoU (Amorphous-EIoU) outperform EIoU on a second amorphous-object domain?**

We test this on **Kvasir-SEG** — 1 000 colonoscopy polyp images from the Simula
Research Lab. Polyps are biomedical objects with highly irregular, concave boundaries
and variable aspect ratios, making them an ideal second test case for the AEIoU
λ-rigidity hypothesis:

> *Amorphous labels are imprecise → down-weighting the shape penalty (λ < 1) should
> improve both accuracy and noise robustness.*

### What this notebook does
1. Downloads Kvasir-SEG and converts segmentation masks → YOLO bounding boxes
2. Builds three validation splits: **clean**, **low-noise** (σ=0.02), **high-noise** (σ=0.08)
3. Trains **33 models** (3 EIoU + 10 AEIoU λ values × 3 splits) using YOLO26n
4. Runs **7 quantitative analyses** and **3 validity checks**
5. Produces publication-ready figures saved to `experiments_kvasir/analysis/`

### Cross-dataset validation
If the optimal λ found here matches the optimal λ from notebook 02 (DUO), that is
strong evidence of a domain-agnostic setting for amorphous object detection.


## Section 1 · Environment Setup

**Requirements**
- Google Colab with T4 GPU (or better — A100 recommended for speed)
- ~2–3 hours total runtime for all 33 training runs on T4
- No API keys required — Kvasir-SEG downloads directly from Simula Research Lab

**Runtime estimate per run:** ~3–5 min on T4 (20 epochs, 640 px, nano model)


In [ ]:
# --- Install pinned dependencies
# ultralytics 8.4.9: confirmed working with yolo26n.pt and our monkey-patch
# wandb 0.24.1: optional experiment tracking (silently skipped if no API key)
!pip install --upgrade pip -q
!pip install -U "ultralytics==8.4.9" "wandb==0.24.1" -q
print("Dependencies installed.")


In [ ]:
# --- Idempotent git clone
# Safe to re-run: skips the clone if the repo is already present.
# The repo contains src/losses.py (EIoULoss, AEIoULoss) and the data/ yaml files.
import os, sys

REPO_PATH = "/content/amorphous-yolo"
if not os.path.exists(f"{REPO_PATH}/.git"):
    print("Cloning amorphous-yolo...")
    os.system(f"git clone https://github.com/nipun-taneja/amorphous-yolo.git {REPO_PATH}")
else:
    print("Repo already present — skipping clone.")

# Add project root to Python path so src.losses is importable
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")


In [ ]:
# --- All experiment constants (single source of truth for this notebook)
import math
from pathlib import Path
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_DIR  = Path("/content/amorphous-yolo")
# Kvasir-SEG dataset root; the zip extracts to kvasir-seg/images/ and kvasir-seg/masks/
DATASET_ROOT = PROJECT_DIR / "datasets" / "kvasir-seg"
# Separate experiments dir from DUO (notebook 02) to avoid run-name collisions
EXPERIMENTS  = PROJECT_DIR / "experiments_kvasir"
ANALYSIS_DIR = EXPERIMENTS / "analysis"
MANIFEST_PATH = EXPERIMENTS / "manifest.json"
EXPERIMENTS.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

# ── Google Drive persistence ───────────────────────────────────────────────────
# Results are synced to Drive after EVERY run so a Colab timeout only loses
# the run currently in progress — all completed runs are safe.
# On session restart, restore_from_drive() copies completed runs back so the
# skip-on-existing logic prevents re-running them.
DRIVE_ROOT        = Path("/content/drive/MyDrive/amorphous_yolo")
DRIVE_EXPERIMENTS = DRIVE_ROOT / "experiments_kvasir"
DRIVE_AVAILABLE   = False   # set to True by mount_drive() below

# ── Training hyper-parameters ─────────────────────────────────────────────────
EPOCHS   = 20     # Sufficient for convergence comparison at nano scale
IMGSZ    = 640    # Standard YOLO input resolution
DEVICE   = 0      # GPU 0; change to "cpu" for debugging
MODEL_PT = "yolo26n.pt"  # Nano model — fast iteration, still meaningful metrics

# ── AEIoU rigidity grid ───────────────────────────────────────────────────────
# λ=0.1 → nearly pure center-alignment loss (shape penalty down-weighted 90%)
# λ=0.5 → moderate trust in polyp extent labels
# λ=1.0 → full size penalty active (normalisation still differs from EIoU — target
#           dims vs enclosing dims — so AEIoU(λ=1) ≠ EIoU; used as cross-check)
AEIOU_RIGIDITIES = [round(x * 0.1, 1) for x in range(1, 11)]

def _fmt_r(r):
    # Format rigidity float for use in run names: 0.3 -> '0p3'
    return str(r).replace(".", "p")

# ── Dataset split configs ─────────────────────────────────────────────────────
# Three validation conditions test model robustness to annotation imprecision.
# Train split is always clean; only the val labels are perturbed.
SPLIT_CONFIGS = {
    "clean": PROJECT_DIR / "data" / "kvasir_seg.yaml",
    "low":   PROJECT_DIR / "data" / "kvasir_seg_low.yaml",
    "high":  PROJECT_DIR / "data" / "kvasir_seg_high.yaml",
}

# ── Visualisation palette ─────────────────────────────────────────────────────
# Colors per loss for consistent plotting throughout all figures
PALETTE = {
    "eiou":        "#E63946",  # Red — EIoU baseline
    "aeiou_r0p1":  "#457B9D",  # Dark blue — λ=0.1
    "aeiou_r0p3":  "#2A9D8F",  # Teal — λ=0.3 (expected winner)
    "aeiou_r0p5":  "#F4A261",  # Orange — λ=0.5
    "aeiou_r1p0":  "#6A4C93",  # Purple — λ=1.0 (EIoU-equivalent rigidity)
}

print("Constants loaded.")
print(f"  AEIOU_RIGIDITIES = {AEIOU_RIGIDITIES}")
print(f"  Experiments dir  : {EXPERIMENTS}")
print(f"  Total planned runs: {1 * len(SPLIT_CONFIGS) + len(AEIOU_RIGIDITIES) * len(SPLIT_CONFIGS)}")


## Section 2 · Kvasir-SEG Dataset

### Why Kvasir-SEG for AEIoU validation?

Kvasir-SEG (Jha et al., 2020) contains **1 000 colonoscopy images** of colorectal
polyps annotated with pixel-level segmentation masks. Key properties that make it
ideal for testing AEIoU:

| Property | Value |
|---|---|
| Images | 1 000 |
| Classes | 1 (polyp) |
| Annotation type | Segmentation mask → converted to bbox |
| Avg polyp area | ~20–60% of image |
| Shape variability | High — round, elongated, flat, pedunculated |
| Boundary regularity | Low — irregular, lobular, concave edges |

### Amorphousness argument
A gastroenterologist annotating a polyp draws a rough contour around a blob with
no fixed shape. The resulting bounding box extent is **annotation-dependent**, not
geometry-determined. Two annotators will produce different box sizes for the same
polyp. This is exactly the regime where λ < 1 in AEIoU should help:
setting λ=0.3 tells the loss "the box center is reliable, but the width/height is
only 30% trustworthy."

### Data preparation pipeline
```
kvasir-seg.zip  →  images/ + masks/
                          ↓  (Cell 8: cv2.findContours + boundingRect)
                     YOLO labels: 0 cx cy w h  (normalised)
                          ↓  (80/20 split)
                     train/  +  valid/
                          ↓  (Cell 9: Gaussian jitter)
                     valid_low/ (σ=0.02)  +  valid_high/ (σ=0.08)
```

**What to look for:** Cell 10 should show 800 train images and 200 val images in each
of the three splits. If counts differ, the split or download step failed.


In [ ]:
# --- WandB setup — proper login with project configuration
# WANDB_API_KEY must be set as a Colab secret (Secrets panel, left sidebar).
# If not set, WandB is disabled so training still runs without hanging.
import os, wandb

# Project name used for all runs in this notebook
WANDB_PROJECT = "amorphous-yolo-kvasir"

# Try to read API key from Colab secrets first (most secure), then fall back
# to an existing environment variable set before this cell ran.
try:
    from google.colab import userdata
    api_key = userdata.get("WANDB_API_KEY")
    if api_key:
        os.environ["WANDB_API_KEY"] = api_key
        print("WandB API key loaded from Colab secrets.")
except Exception:
    pass   # not in Colab or secret not set

if os.environ.get("WANDB_API_KEY"):
    wandb.login(key=os.environ["WANDB_API_KEY"], relogin=False)
    print(f"WandB logged in. Project: {WANDB_PROJECT}")
else:
    os.environ["WANDB_MODE"] = "disabled"
    print("WANDB_API_KEY not found — WandB disabled.")
    print("To enable: add WANDB_API_KEY in Colab Secrets (key icon, left sidebar).")


### Google Drive: Mount, Restore & Persist

Results are written to Drive **after every training run** so a Colab timeout
only loses the run currently in progress. On session restart, completed runs
are restored from Drive so the notebook resumes exactly where it left off.

**What to look for:** The restore step should print a list of run names copied
back from Drive. These runs will show `[SKIP]` in the training cells below,
meaning no computation is wasted repeating them.

**Setup:** Drive will be mounted automatically below. If it fails (e.g. running
locally), training still works — Drive sync is silently skipped.


In [ ]:
# --- Google Drive: mount + restore completed runs from previous sessions
import shutil

def mount_drive():
    # Mount Google Drive and set DRIVE_AVAILABLE = True if successful.
    global DRIVE_AVAILABLE
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_EXPERIMENTS.mkdir(parents=True, exist_ok=True)
        DRIVE_AVAILABLE = True
        print(f"Drive mounted. Backup dir: {DRIVE_EXPERIMENTS}")
    except Exception as e:
        print(f"Drive not available ({e}). Running without Drive persistence.")
        DRIVE_AVAILABLE = False
    return DRIVE_AVAILABLE


def restore_from_drive():
    # Copy completed runs from Drive back to local EXPERIMENTS dir.
    # Called once at session start so skip-on-existing logic works correctly
    # even after a Colab timeout/restart.
    if not DRIVE_AVAILABLE:
        return
    if not DRIVE_EXPERIMENTS.exists():
        print("No Drive backup found yet — starting fresh.")
        return
    restored = 0
    for drive_run in sorted(DRIVE_EXPERIMENTS.iterdir()):
        if not drive_run.is_dir():
            continue
        local_run = EXPERIMENTS / drive_run.name
        # Only restore runs that completed (have results.csv) and aren't already local
        if (drive_run / "results.csv").exists() and not (local_run / "results.csv").exists():
            shutil.copytree(str(drive_run), str(local_run), dirs_exist_ok=True)
            restored += 1
            print(f"  [RESTORE] {drive_run.name}")
    if restored == 0:
        print("Nothing to restore — local experiments are up to date.")
    else:
        print(f"Restored {restored} completed run(s) from Drive.")


mount_drive()
restore_from_drive()


In [ ]:
# --- Download Kvasir-SEG (idempotent)
# Source: Simula Research Lab official server — no API key required.
# File size: ~44 MB zip. Extracts to datasets/kvasir-seg/images/ and masks/.
import zipfile, urllib.request, shutil

kvasir_zip  = PROJECT_DIR / "datasets" / "kvasir-seg.zip"
images_dir  = DATASET_ROOT / "images"

if images_dir.exists() and len(list(images_dir.glob("*.jpg"))) > 900:
    print(f"Kvasir-SEG already present ({len(list(images_dir.glob('*.jpg')))} images) — skipping download.")
else:
    print("Downloading Kvasir-SEG (~44 MB)...")
    (PROJECT_DIR / "datasets").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(
        "https://datasets.simula.no/downloads/kvasir-seg.zip",
        kvasir_zip,
    )
    print(f"  Downloaded → {kvasir_zip}")

    # Extract: creates kvasir-seg/ folder inside datasets/
    with zipfile.ZipFile(kvasir_zip, "r") as z:
        z.extractall(PROJECT_DIR / "datasets")
    print(f"  Extracted  → {DATASET_ROOT}")

    # Verify extracted structure
    n_imgs  = len(list((DATASET_ROOT / "images").glob("*.jpg")))
    n_masks = len(list((DATASET_ROOT / "masks").glob("*.jpg")))
    print(f"  Images: {n_imgs}  |  Masks: {n_masks}")
    assert n_imgs == n_masks == 1000, f"Expected 1000 pairs, got {n_imgs}/{n_masks}"
    print("Download complete.")


In [ ]:
# --- Convert segmentation masks -> YOLO bbox labels + 80/20 train/val split
# Kvasir-SEG provides PNG segmentation masks (white=polyp, black=background).
# We find the bounding box of all non-zero pixels using cv2.findContours,
# then write a YOLO label file: "0 cx cy w h" (all normalised to [0,1]).
#
# Each image has at most one polyp region (single-class dataset), so each
# label file contains exactly one line.
import cv2
import numpy as np
import random
import shutil

TRAIN_DIR  = DATASET_ROOT / "train"
VALID_DIR  = DATASET_ROOT / "valid"

def _mask_to_yolo_bbox(mask_path):
    # Load a Kvasir-SEG mask and return (cx, cy, w, h) normalised to [0,1].
    # Returns None if the mask is blank (no polyp found).
    # Read mask as grayscale; Kvasir masks are 3-channel JPEGs with white polyp
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None

    # Threshold to binary: pixel > 127 → foreground
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # Find contours of the polyp region(s)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None  # Blank mask — skip this image

    # Union bounding rect across all contours (handles multi-fragment masks)
    all_pts = np.vstack(contours)  # shape: [N_pts, 1, 2]
    x, y, w, h = cv2.boundingRect(all_pts)

    # Normalise by image dimensions (H, W)
    H, W = mask.shape
    cx = (x + w / 2) / W
    cy = (y + h / 2) / H
    bw = w / W
    bh = h / H

    # Clamp to [0, 1] for safety (rounding at image edges)
    cx, cy, bw, bh = [float(np.clip(v, 0.0, 1.0)) for v in [cx, cy, bw, bh]]
    return cx, cy, bw, bh


if (TRAIN_DIR / "images").exists() and len(list((TRAIN_DIR/"images").glob("*.jpg"))) >= 800:
    print("Train/val split already exists — skipping conversion.")
else:
    print("Converting masks → YOLO labels and creating train/val split...")
    for split_dir in [TRAIN_DIR, VALID_DIR]:
        (split_dir / "images").mkdir(parents=True, exist_ok=True)
        (split_dir / "labels").mkdir(parents=True, exist_ok=True)

    # Collect all image stems and shuffle for reproducible split
    all_imgs = sorted((DATASET_ROOT / "images").glob("*.jpg"))
    random.seed(42)
    random.shuffle(all_imgs)

    # 80% train, 20% val
    split_idx = int(0.8 * len(all_imgs))
    splits = {"train": all_imgs[:split_idx], "valid": all_imgs[split_idx:]}

    skipped = 0
    for split_name, img_list in splits.items():
        out_img_dir = DATASET_ROOT / split_name / "images"
        out_lbl_dir = DATASET_ROOT / split_name / "labels"
        for img_path in img_list:
            stem = img_path.stem
            # Kvasir masks share the same filename as images (jpg)
            mask_path = DATASET_ROOT / "masks" / f"{stem}.jpg"
            bbox = _mask_to_yolo_bbox(mask_path)
            if bbox is None:
                skipped += 1
                continue
            cx, cy, bw, bh = bbox
            # Copy image (hard copy so paths are self-contained)
            shutil.copy2(img_path, out_img_dir / img_path.name)
            # Write YOLO label: class_id cx cy w h
            (out_lbl_dir / f"{stem}.txt").write_text(
                f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}
"
            )

    print(f"  Conversion complete. Skipped {skipped} blank masks.")
    print(f"  Train: {len(list((TRAIN_DIR/'images').glob('*.jpg')))} images")
    print(f"  Valid: {len(list((VALID_DIR/'images').glob('*.jpg')))} images")


In [ ]:
# --- Build noise-perturbed validation splits (idempotent)
# Applies Gaussian jitter to YOLO bbox coordinates in valid/labels/ to simulate
# annotation noise. Unlike DUO (polygon format), Kvasir labels are standard
# axis-aligned YOLO boxes so jitter is applied directly to (cx, cy, w, h).
#
# sigma_low  = 0.02: ≈2% of image dimension → mild annotation imprecision
# sigma_high = 0.08: ≈8% of image dimension → severe noise, stress-test
import numpy as np
import shutil
from pathlib import Path

SIGMA_LOW  = 0.02
SIGMA_HIGH = 0.08

def _jitter_label_file(src_lbl, dst_lbl, sigma, rng):
    """Perturb all bbox coords in a YOLO label file by N(0, sigma) and clip to [0,1]."""
    lines = Path(src_lbl).read_text().strip().split("\n")
    out = []
    for line in lines:
        if not line.strip():
            continue
        parts = line.split()
        cls_id = parts[0]
        cx, cy, w, h = [float(v) for v in parts[1:5]]
        # Add independent Gaussian noise to each coordinate
        # Clamp: ensures boxes stay inside image boundaries
        cx = float(np.clip(cx + rng.normal(0, sigma), 0.0, 1.0))
        cy = float(np.clip(cy + rng.normal(0, sigma), 0.0, 1.0))
        w  = float(np.clip(w  + rng.normal(0, sigma), 0.01, 1.0))  # min 1% width
        h  = float(np.clip(h  + rng.normal(0, sigma), 0.01, 1.0))  # min 1% height
        out.append(f"{cls_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    Path(dst_lbl).write_text("\n".join(out) + "\n")


def build_kvasir_noise_splits(root: Path, seed: int = 42):
    """Build valid_low/ and valid_high/ from valid/. Idempotent."""
    src_img_dir = root / "valid" / "images"
    src_lbl_dir = root / "valid" / "labels"
    rng = np.random.default_rng(seed)

    for split_name, sigma in [("valid_low", SIGMA_LOW), ("valid_high", SIGMA_HIGH)]:
        dst_img_dir = root / split_name / "images"
        dst_lbl_dir = root / split_name / "labels"

        if dst_img_dir.exists() and len(list(dst_img_dir.glob("*.jpg"))) >= 200:
            print(f"  {split_name}: already exists — skipping.")
            continue

        dst_img_dir.mkdir(parents=True, exist_ok=True)
        dst_lbl_dir.mkdir(parents=True, exist_ok=True)

        for img_path in sorted(src_img_dir.glob("*.jpg")):
            stem = img_path.stem
            # Images are identical — only labels change
            # Use symlink where possible; fall back to copy on Windows
            dst_img = dst_img_dir / img_path.name
            if not dst_img.exists():
                try:
                    dst_img.symlink_to(img_path)
                except (OSError, NotImplementedError):
                    shutil.copy2(img_path, dst_img)

            src_lbl = src_lbl_dir / f"{stem}.txt"
            dst_lbl = dst_lbl_dir / f"{stem}.txt"
            if src_lbl.exists():
                _jitter_label_file(src_lbl, dst_lbl, sigma, rng)

        n = len(list(dst_img_dir.glob("*.jpg")))
        print(f"  {split_name}: {n} images (σ={sigma})")


build_kvasir_noise_splits(DATASET_ROOT)
print("Noise splits ready.")


In [ ]:
# --- Verify all three split configs exist and have correct image counts
import yaml

print(f"{'Split':<10} {'YAML':<40} {'Val images':>12}")
print("-" * 65)
for split_name, cfg_path in SPLIT_CONFIGS.items():
    assert cfg_path.exists(), f"Missing yaml: {cfg_path}"
    cfg = yaml.safe_load(cfg_path.read_text())
    val_dir = DATASET_ROOT / cfg["val"]
    n_imgs = len(list(val_dir.glob("*.jpg")))
    status = "✓" if n_imgs >= 200 else "✗ CHECK"
    print(f"{split_name:<10} {str(cfg_path.name):<40} {n_imgs:>10}  {status}")

# Training split (same for all configs)
n_train = len(list((DATASET_ROOT / "train" / "images").glob("*.jpg")))
print(f"\nTrain images (shared): {n_train}")
assert n_train >= 800, f"Expected 800 train images, found {n_train}"
print("All splits verified.")


### Kvasir-SEG Sample Images

The following cell displays 5 random training images with ground-truth bounding boxes
overlaid in green. Labels show normalised box dimensions.

**What to look for:**
- Polyps should vary in size from small (<10% of image area) to large (>50%)
- Some polyps are near-circular; others are elongated or irregular blobs
- No two polyps look the same — this confirms the amorphous-object assumption
- The GT box should tightly contain the polyp with minimal slack

If boxes look misaligned, the mask→bbox conversion in Cell 8 may have a coordinate error.


In [ ]:
# --- Visualise 5 random training images with GT bounding boxes
# This is a qualitative sanity check: do the labels look correct?
import cv2, random, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def _draw_yolo_bbox(ax, label_path, img_w, img_h, color="lime", lw=2):
    """Draw YOLO-format bbox(es) from a label file onto a matplotlib axis."""
    if not Path(label_path).exists():
        return
    for line in Path(label_path).read_text().strip().split("\n"):
        if not line.strip():
            continue
        _, cx, cy, bw, bh = [float(v) for v in line.split()]
        # Convert normalised (cx,cy,w,h) → pixel (x1,y1,w,h) for matplotlib
        x1 = (cx - bw / 2) * img_w
        y1 = (cy - bh / 2) * img_h
        rect = patches.Rectangle(
            (x1, y1), bw * img_w, bh * img_h,
            linewidth=lw, edgecolor=color, facecolor="none",
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"polyp  w={bw:.2f} h={bh:.2f}",
                color=color, fontsize=7, fontweight="bold",
                bbox=dict(facecolor="black", alpha=0.5, pad=1))


train_imgs = sorted((DATASET_ROOT / "train" / "images").glob("*.jpg"))
sample = random.sample(train_imgs, 5)

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
fig.suptitle("Kvasir-SEG — 5 random training images with GT bounding boxes",
             fontsize=13, fontweight="bold")

for ax, img_path in zip(axes, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    ax.imshow(img)
    lbl_path = DATASET_ROOT / "train" / "labels" / f"{img_path.stem}.txt"
    _draw_yolo_bbox(ax, lbl_path, W, H)
    ax.set_title(img_path.stem[:20], fontsize=8)
    ax.axis("off")

plt.tight_layout()
save_path = ANALYSIS_DIR / "00_sample_images.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/00_sample_images.png
plt.show()
print(f"Saved → {save_path}")


**Figure 0 · Sample Kvasir-SEG Training Images**

*x-axis / y-axis:* pixel coordinates · *Green boxes:* ground-truth YOLO bounding boxes
*Labels:* show normalised width and height of each box

**Expected pattern:** Boxes of varying sizes and aspect ratios — small flat lesions
alongside large pedunculated polyps. Irregular polyp shapes that extend to one side of
the box are normal and confirm the amorphous assumption.
If all boxes appear the same size/shape, the mask conversion may have a bug.


## Section 3 · Theory: EIoU vs AEIoU

### Formula Comparison

$$L_{\text{EIoU}} = (1-\text{IoU}) + \frac{\rho^2(b,b^{gt})}{c^2} + \frac{(p_w - t_w)^2}{c_w^2} + \frac{(p_h - t_h)^2}{c_h^2}$$

$$L_{\text{AEIoU}} = (1-\text{IoU}) + \frac{\rho^2(b,b^{gt})}{c^2} + \lambda \left(\frac{(p_w - t_w)^2}{t_w^2} + \frac{(p_h - t_h)^2}{t_h^2}\right)$$

| Component | EIoU | AEIoU |
|---|---|---|
| Overlap term | $1-\text{IoU}$ | $1-\text{IoU}$ (identical) |
| Center term | $\rho^2/c^2$ | $\rho^2/c^2$ (identical) |
| Width normaliser | $c_w^2$ (enclosing width) | $t_w^2$ (target width) |
| Height normaliser | $c_h^2$ (enclosing height) | $t_h^2$ (target height) |
| Shape weight | Fixed = 1 | Tunable λ ∈ [0, 1] |

### Why the normaliser matters

**EIoU** normalises size errors by the enclosing box dimensions ($c_w$, $c_h$).
The enclosing box grows with scene clutter — if nearby objects exist, $c_w$ is large
and the width-error penalty is *lenient*. This is sensible for rigid objects in
cluttered scenes: a 10 px misalignment on a car doesn't matter if the scene context
is 200 px wide.

**AEIoU** normalises by the *target* dimensions ($t_w$, $t_h$).
A 10 px error on a 20 px polyp is penalised as 25% of target width squared.
The same error on a 200 px polyp is penalised as 0.25% — it's *label-relative*,
not context-relative. This is more natural for amorphous objects where the target
label is the only reliable scale reference.

### The λ-rigidity argument for polyps

A gastroenterologist annotating a polyp draws a rough contour. The resulting
bounding box **extent** depends on where they stopped drawing, not just polyp size.
Two clinicians annotating the same polyp will produce different box widths/heights.

Setting λ < 1 says: *"I trust the center coordinates more than the extent."*
- λ = 0.1: near-pure center-alignment loss — ignore width/height error completely
- λ = 0.3: down-weight size penalty 70% — moderate scepticism about extent labels
- λ = 1.0: full size penalty — same strength as EIoU (but different normaliser)

**Prediction:** The optimal λ for Kvasir-SEG will be 0.2–0.4, matching the DUO
dataset result, suggesting a domain-agnostic optimal λ for amorphous objects.

**What to look for in Cells 14–15:** Loss values should be monotonically decreasing
from no-overlap → perfect-overlap scenarios. AEIoU(λ=0.1) should produce the lowest
absolute loss values (smallest shape penalty), while EIoU and AEIoU(λ=1.0) should
produce the highest values when boxes differ in size.


In [ ]:
# --- Numerical comparison: EIoU vs AEIoU on 3 synthetic scenarios
# This is a unit-test style check: verify that the losses respond correctly
# to three canonical prediction/target configurations.
import torch
from src.losses import EIoULoss, AEIoULoss

eiou_fn  = EIoULoss(reduction="none")
aeiou_fn = {r: AEIoULoss(rigidity=r, reduction="none") for r in [0.1, 0.3, 0.5, 1.0]}

# Three canonical scenarios (xyxy format, pixel coordinates)
scenarios = {
    "Perfect match":   (torch.tensor([[10.,10.,50.,50.]]), torch.tensor([[10.,10.,50.,50.]])),
    "Partial overlap": (torch.tensor([[10.,10.,40.,40.]]), torch.tensor([[20.,20.,50.,50.]])),
    "No overlap":      (torch.tensor([[10.,10.,30.,30.]]), torch.tensor([[40.,40.,60.,60.]])),
}

# Build comparison table
header = f"{'Scenario':<20} {'EIoU':>8}" + "".join(f" AEIoU λ={r:>3}":>12 for r in [0.1, 0.3, 0.5, 1.0])
print(header)
print("-" * len(header))

for scenario_name, (pred, target) in scenarios.items():
    eiou_val = eiou_fn(pred, target).item()
    row = f"{scenario_name:<20} {eiou_val:>8.4f}"
    for r, fn in aeiou_fn.items():
        val = fn(pred, target).item()
        row += f" {val:>12.4f}"
    print(row)

print("\nExpected: loss decreases monotonically from 'No overlap' → 'Perfect match'.")
print("AEIoU(λ=0.1) should be lowest on 'Partial overlap' (shape penalty suppressed).")


**Table · Numerical Loss Comparison**

*Rows:* three canonical prediction/target configurations
*Columns:* EIoU and AEIoU at four λ values

**Expected pattern:** All losses = 0.0 on "Perfect match", highest on "No overlap".
On "Partial overlap", AEIoU(λ=0.1) should give the lowest value because the
width/height error penalty is nearly zero, leaving only overlap + center terms.
AEIoU(λ=1.0) should be close to but not equal to EIoU because it uses target
dims rather than enclosing dims as the normaliser.


In [ ]:
# --- Loss sensitivity sweep: slide from no-overlap to perfect overlap
# Constructs a 1D sweep where the predicted box moves from no overlap (left)
# to perfect overlap (right) with the target. Plots loss value vs IoU.
# This reveals the gradient landscape each loss provides to the optimiser.
import matplotlib.pyplot as plt
import torch
import numpy as np

target = torch.tensor([[40., 40., 80., 80.]])  # Fixed target: 40×40 px box

# Sweep: predicted box x-offset from -40 (no overlap) to 0 (perfect overlap)
offsets = np.linspace(-40, 0, 200)
ious, eiou_vals = [], []
aeiou_vals = {r: [] for r in [0.1, 0.3, 0.5, 1.0]}

for dx in offsets:
    # Predicted box shifted right by dx relative to no-overlap position
    pred = torch.tensor([[dx + 40., 40., dx + 80., 80.]])
    pred = pred.clamp(min=0)

    # Compute IoU for x-axis
    inter_x = max(0, min(pred[0,2], target[0,2]) - max(pred[0,0], target[0,0]))
    inter_y = max(0, min(pred[0,3], target[0,3]) - max(pred[0,1], target[0,1]))
    inter = inter_x * inter_y
    area_p = (pred[0,2]-pred[0,0]) * (pred[0,3]-pred[0,1])
    area_t = (target[0,2]-target[0,0]) * (target[0,3]-target[0,1])
    iou = inter / (area_p + area_t - inter + 1e-7)
    ious.append(float(iou))

    eiou_vals.append(float(EIoULoss(reduction="none")(pred, target)))
    for r in [0.1, 0.3, 0.5, 1.0]:
        aeiou_vals[r].append(float(AEIoULoss(rigidity=r, reduction="none")(pred, target)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ious, eiou_vals, color="#E63946", lw=2.5, label="EIoU", zorder=5)
for r, color in zip([0.1, 0.3, 0.5, 1.0], ["#457B9D","#2A9D8F","#F4A261","#6A4C93"]):
    ax.plot(ious, aeiou_vals[r], color=color, lw=1.8, linestyle="--",
            label=f"AEIoU λ={r}")

ax.set_xlabel("IoU with target box", fontsize=12)
ax.set_ylabel("Loss value", fontsize=12)
ax.set_title("Loss sensitivity: no-overlap → perfect overlap", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=10)
ax.invert_xaxis()  # left=no overlap (IoU=0), right=perfect (IoU=1)
ax.grid(alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "01_loss_sensitivity.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/01_loss_sensitivity.png
plt.show()
print(f"Saved → {save_path}")


**Figure 1 · Loss Sensitivity Sweep**

*x-axis:* IoU between predicted and target box (right = perfect overlap, left = no overlap)
*y-axis:* scalar loss value · *Solid red:* EIoU · *Dashed:* AEIoU at four λ values

**Expected pattern:**
- All curves converge to 0 at IoU = 1.0 (rightmost point)
- At IoU = 0 (leftmost), AEIoU(λ=0.1) has the smallest loss because the size penalty is
  nearly zero — only the center-distance and overlap terms remain active
- EIoU and AEIoU(λ=1.0) should have the highest loss at IoU = 0
- The spread between curves shrinks as boxes overlap more (size error → 0)

If AEIoU(λ=1.0) ≈ EIoU throughout, the two normalisers (target vs enclosing) have
negligible effect — that would *weaken* the AEIoU novelty claim.


## Section 4 · Monkey-Patch Infrastructure

### Why monkey-patch instead of editing Ultralytics source?

Ultralytics is a versioned pip package. Editing its source would:
1. Break on `pip install --upgrade ultralytics`
2. Make the experiment hard to reproduce on a fresh Colab runtime
3. Require maintaining a fork

Instead, we replace `BboxLoss.forward` at runtime with a closure that:
- Calls our custom loss function for the **IoU term**
- Copies the **DFL term** verbatim from Ultralytics 8.4.9
- Is fully reversible via `restore_loss()` — critical between runs

The patch is applied immediately before `model.train()` and removed in a
`try/finally` block so it is guaranteed to be restored even if training crashes.

**What to look for in Cell 18:** Running `patch_loss(EIoULoss(...))` then
`restore_loss()` should leave `BboxLoss.forward` identical to its original form
(same function object reference). The print at the end confirms this.


In [ ]:
# --- Full monkey-patch implementation (verbatim pattern from notebook 02)
# BboxLoss.forward signature in ultralytics 8.4.9:
#   forward(self, pred_dist, pred_bboxes, anchor_points,
#           target_bboxes, target_scores, target_scores_sum, fg_mask, imgsz, stride)
import types
import torch
import torch.nn.functional as F
import ultralytics.utils.loss as loss_mod

# Save the original forward method ONCE at import time.
# This is the reference we restore after each training run.
_ORIGINAL_BBOX_FORWARD = loss_mod.BboxLoss.forward


def _make_bbox_forward(loss_fn_instance):
    """
    Factory: returns a bound-method replacement for BboxLoss.forward that
    uses loss_fn_instance for the IoU term and keeps DFL unchanged.

    loss_fn_instance must support reduction='none' and return shape [N].
    """
    def bbox_loss_forward(
        self,
        pred_dist, pred_bboxes, anchor_points,
        target_bboxes, target_scores, target_scores_sum,
        fg_mask, imgsz, stride,
    ):
        # ── IoU term ──────────────────────────────────────────────────────────
        # target_scores: [B, A, C] — score per anchor; sum over classes gives weight
        weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)  # shape: [N, 1]

        # Apply our custom loss to foreground anchors only (fg_mask selects them)
        # pred_bboxes and target_bboxes are in xyxy format, normalised to image size
        per_box = loss_fn_instance(
            pred_bboxes[fg_mask],    # [N, 4] xyxy predicted
            target_bboxes[fg_mask],  # [N, 4] xyxy ground truth
        )  # → [N] per-box loss values (reduction='none')

        # Weighted average over foreground anchors, normalised by total score sum
        loss_iou = (per_box.unsqueeze(-1) * weight).sum() / target_scores_sum

        # ── DFL term (copied verbatim from ultralytics 8.4.9) ─────────────────
        # DFL (Distribution Focal Loss) operates on the predicted distance
        # distribution; it is independent of the IoU metric choice.
        if self.dfl_loss:
            target_ltrb = loss_mod.bbox2dist(
                anchor_points, target_bboxes, self.dfl_loss.reg_max - 1
            )
            loss_dfl = self.dfl_loss(
                pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max),
                target_ltrb[fg_mask],
            ) * weight
            loss_dfl = loss_dfl.sum() / target_scores_sum
        else:
            target_ltrb = loss_mod.bbox2dist(anchor_points, target_bboxes)
            target_ltrb = target_ltrb * stride
            target_ltrb[..., 0::2] /= imgsz[1]
            target_ltrb[..., 1::2] /= imgsz[0]
            pred_dist_s = pred_dist * stride
            pred_dist_s[..., 0::2] /= imgsz[1]
            pred_dist_s[..., 1::2] /= imgsz[0]
            loss_dfl = (
                F.l1_loss(pred_dist_s[fg_mask], target_ltrb[fg_mask],
                          reduction="none").mean(-1, keepdim=True) * weight
            )
            loss_dfl = loss_dfl.sum() / target_scores_sum

        return loss_iou, loss_dfl

    return bbox_loss_forward


def patch_loss(loss_fn_instance):
    """Apply custom loss patch. Always call restore_loss() before patching again."""
    loss_mod.BboxLoss.forward = _make_bbox_forward(loss_fn_instance)
    print(f"  [PATCH] BboxLoss.forward → {type(loss_fn_instance).__name__}"
          + (f"(λ={loss_fn_instance.rigidity})" if hasattr(loss_fn_instance, "rigidity") else ""))


def restore_loss():
    """Restore the original Ultralytics CIoU-based BboxLoss.forward."""
    loss_mod.BboxLoss.forward = _ORIGINAL_BBOX_FORWARD


print("Patch infrastructure ready.")
print(f"  Original forward saved: {_ORIGINAL_BBOX_FORWARD}")


In [ ]:
# --- Verify patch round-trip
from src.losses import EIoULoss

# Step 1: apply patch
patch_loss(EIoULoss(reduction="none"))
patched_forward = loss_mod.BboxLoss.forward
print(f"After patch:   BboxLoss.forward is <function bbox_loss_forward>: "
      f"{'bbox_loss_forward' in str(patched_forward)}")

# Step 2: restore
restore_loss()
restored_forward = loss_mod.BboxLoss.forward
print(f"After restore: BboxLoss.forward is original: "
      f"{restored_forward is _ORIGINAL_BBOX_FORWARD}")

assert restored_forward is _ORIGINAL_BBOX_FORWARD, "restore_loss() failed!"
print("Patch round-trip verified.")


## Section 5 · Loss Registry

This notebook uses only **EIoU** and **AEIoU** — a focused comparison against the
single most similar competing loss. The registry pattern (dict of loss instances)
means adding a new loss requires exactly one line.

Note: `AEIOU_LOSS_REGISTRY["aeiou_r1p0"]` (λ=1.0) is the λ=1.0 end of the grid.
Since AEIoU(λ=1.0) uses target-dim normalisation while EIoU uses enclosing-dim
normalisation, they should produce **similar but not identical** mAP — a useful
sanity check confirming the normalisers differ.


In [ ]:
# --- Loss registry: one instance per loss configuration
from src.losses import EIoULoss, AEIoULoss

# Single EIoU instance — this is the baseline for all comparisons
EIOU_INSTANCE = EIoULoss(reduction="none")
print(f"EIoU: {EIOU_INSTANCE}")

# AEIoU grid: 10 instances, one per λ value
# reduction='none' required by the monkey-patch (per-box loss values needed)
AEIOU_LOSS_REGISTRY = {
    f"aeiou_r{_fmt_r(r)}": AEIoULoss(rigidity=r, reduction="none")
    for r in AEIOU_RIGIDITIES
}

print(f"\nAEIoU registry ({len(AEIOU_LOSS_REGISTRY)} instances):")
for name, fn in AEIOU_LOSS_REGISTRY.items():
    print(f"  {name}: rigidity={fn.rigidity}")

print(f"\nTotal loss configs: 1 EIoU + {len(AEIOU_LOSS_REGISTRY)} AEIoU = "
      f"{1 + len(AEIOU_LOSS_REGISTRY)} configs × {len(SPLIT_CONFIGS)} splits = "
      f"{(1 + len(AEIOU_LOSS_REGISTRY)) * len(SPLIT_CONFIGS)} total runs")


## Section 6 · Training Infrastructure

### Run naming convention
```
kvasir_yolo26n_{loss_key}_{split}_e{epochs}
```
Examples:
- `kvasir_yolo26n_eiou_clean_e20`
- `kvasir_yolo26n_aeiou_r0p3_low_e20`
- `kvasir_yolo26n_aeiou_r1p0_high_e20`

`loss_key` always comes from the registry — never constructed ad-hoc.

### Idempotency
`run_training()` checks if `{run_dir}/results.csv` already exists.
If yes → `[SKIP]`. This means any interrupted session can be resumed by
re-running the training cells without duplicating completed runs.

### Manifest
`experiments_kvasir/manifest.json` is updated after **every** run (even failures).
Status field: `"running"` → `"complete"` or `"failed"`. This allows resuming
interrupted sessions and auditing which runs succeeded.

### GPU requirements
- T4 (16 GB): ~3–5 min/run × 33 runs = ~2.5 hrs total
- A100 (40 GB): ~1–2 min/run = ~1 hr total
- Recommended: rent an A100 or L4 for this notebook

**What to look for:** Each run's stdout should show `[START]` then Ultralytics'
training progress bar. `[SKIP]` means the run was already completed — safe to ignore.


In [ ]:
# --- Training functions: run_training + write_manifest_entry + sync_to_drive
import json, shutil
from datetime import datetime
from ultralytics import YOLO

def _load_manifest():
    # Load manifest.json as a dict, or return empty dict if not present.
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {}


def write_manifest_entry(run_name, meta):
    # Atomically update manifest.json with the entry for run_name.
    manifest = _load_manifest()
    manifest[run_name] = meta
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))


def sync_to_drive(run_name):
    # Copy a single completed run directory from local storage to Google Drive.
    # Called in run_training()'s finally block so every run is persisted
    # immediately — a Colab timeout after this point loses nothing.
    if not DRIVE_AVAILABLE:
        return
    local_run = EXPERIMENTS / run_name
    drive_run = DRIVE_EXPERIMENTS / run_name
    try:
        shutil.copytree(str(local_run), str(drive_run), dirs_exist_ok=True)
        print(f"  [DRIVE] Synced {run_name}")
    except Exception as e:
        print(f"  [DRIVE] Sync failed for {run_name}: {e}")


def run_training(loss_name, loss_fn, split_name, yaml_path,
                 epochs=None, imgsz=None, device=None):
    """
    Train one YOLO26n model. Returns the experiment run directory Path.

    Parameters
    ----------
    loss_name  : str        Key from loss registry (e.g. 'eiou', 'aeiou_r0p3')
    loss_fn    : nn.Module  Loss instance to patch into BboxLoss.forward
    split_name : str        'clean' | 'low' | 'high'
    yaml_path  : Path       Dataset config YAML file path
    """
    epochs = epochs if epochs is not None else EPOCHS
    imgsz  = imgsz  if imgsz  is not None else IMGSZ
    device = device if device is not None else DEVICE

    # Canonical run name — deterministic from inputs, never constructed ad-hoc
    run_name = f"kvasir_yolo26n_{loss_name}_{split_name}_e{epochs}"
    run_dir  = EXPERIMENTS / run_name

    # ── Skip if already completed ──────────────────────────────────────────────
    if (run_dir / "results.csv").exists():
        print(f"[SKIP] {run_name}")
        return run_dir

    print(f"\n{'='*70}")
    print(f"[START] {run_name}")
    print(f"  loss={loss_name}  split={split_name}  epochs={epochs}")
    print(f"{'='*70}")

    # Record intent in manifest before training (status='running')
    # This allows detecting crashed runs that never reached status='complete'
    meta = {
        "loss":        loss_name,
        "split":       split_name,
        "epochs":      epochs,
        "rigidity":    float(getattr(loss_fn, "rigidity", -1) or -1),
        "run_dir":     str(run_dir),
        "weights":     str(run_dir / "weights" / "best.pt"),
        "results_csv": str(run_dir / "results.csv"),
        "timestamp":   datetime.now().isoformat(),
        "status":      "running",
    }
    write_manifest_entry(run_name, meta)

    try:
        # Name this run in WandB before training starts so it appears correctly
        # in the dashboard even if the session is interrupted mid-run.
        import os as _os
        _os.environ["WANDB_PROJECT"] = WANDB_PROJECT
        _os.environ["WANDB_NAME"]    = run_name
        _os.environ["WANDB_TAGS"]    = f"{loss_name},{split_name}"

        # Apply our custom loss to BboxLoss.forward
        patch_loss(loss_fn)

        model = YOLO(MODEL_PT)
        results = model.train(
            data=str(yaml_path),
            epochs=epochs,
            imgsz=imgsz,
            project=str(EXPERIMENTS),
            name=run_name,
            device=device,
            exist_ok=True,
        )

        # Snapshot Ultralytics metadata for offline analysis
        try:
            (run_dir / "run_meta.json").write_text(
                json.dumps(results.results_dict, indent=2)
            )
        except Exception as e:
            print(f"  [WARN] Could not write run_meta.json: {e}")

        meta["status"] = "complete"

        # Explicitly finish the WandB run so metrics are fully uploaded
        # before we move to the next training run.
        try:
            import wandb as _wandb
            if _wandb.run is not None:
                _wandb.finish()
        except Exception:
            pass

    except Exception as e:
        print(f"  [ERROR] {run_name} failed: {e}")
        meta["status"] = "failed"
        meta["error"]  = str(e)
        raise

    finally:
        # CRITICAL: always restore — never leave Ultralytics in patched state
        restore_loss()
        write_manifest_entry(run_name, meta)
        # Sync this run to Drive immediately — protects against future timeouts.
        # A Colab crash after this line loses at most the next run, never this one.
        sync_to_drive(run_name)

    print(f"[DONE] {run_name}")
    return run_dir


print("run_training() ready.")
print(f"Manifest path: {MANIFEST_PATH}")


## Section 7 · EIoU Baseline Training (3 runs)

| Run | Split | Purpose |
|---|---|---|
| `kvasir_yolo26n_eiou_clean_e20` | clean | EIoU upper-bound mAP |
| `kvasir_yolo26n_eiou_low_e20` | low (σ=0.02) | EIoU under mild noise |
| `kvasir_yolo26n_eiou_high_e20` | high (σ=0.08) | EIoU under severe noise |

These three runs form the **baseline** against which all 30 AEIoU runs are compared.
The gap between `clean` and `high` mAP is the **EIoU robustness gap** — we expect
AEIoU (especially low λ) to have a smaller gap.

**Expected runtime:** ~10–15 minutes for all 3 runs on T4.


In [ ]:
# --- EIoU baseline training (3 runs × 1 split config each)
# Re-running this cell is safe — completed runs are skipped via [SKIP] logic.
for split_name, cfg_path in SPLIT_CONFIGS.items():
    run_training(
        loss_name="eiou",
        loss_fn=EIOU_INSTANCE,
        split_name=split_name,
        yaml_path=cfg_path,
    )

# Defensive check: restore should already be called in finally block above,
# but call again to be absolutely safe before moving to AEIoU runs.
restore_loss()
print("\nAll EIoU baseline runs complete (or skipped).")


## Section 8 · AEIoU Rigidity Grid Search (30 runs)

We sweep λ from 0.1 to 1.0 in steps of 0.1 across all three splits:
- 10 λ values × 3 splits = **30 training runs**

### λ interpretation for polyp detection

| λ | Interpretation | Expected behaviour |
|---|---|---|
| 0.1 | Shape labels nearly ignored — pure center+overlap loss | High robustness, possibly lower clean mAP |
| 0.2–0.4 | Moderate shape trust — expected optimal range | Best clean mAP, good robustness |
| 0.5–0.7 | Stronger shape penalty | Converges to EIoU-like behaviour |
| 1.0 | Full shape penalty (different normaliser than EIoU) | ~= EIoU, confirms normaliser effect is small |

**Note:** `aeiou_r1p0` and `eiou` differ only in the size-term normaliser
(target dims vs enclosing dims). If their mAP is nearly identical, the normaliser
choice has minimal practical impact. If `aeiou_r1p0` noticeably underperforms `eiou`
on clean data, the target-dim normaliser may be too aggressive.

**Expected runtime:** ~90–150 minutes for all 30 runs on T4.
Grab a coffee. Or run overnight with a Colab Pro session.


In [ ]:
# --- AEIoU rigidity grid training (30 runs)
# Outer loop: rigidity values (0.1 → 1.0)
# Inner loop: splits (clean, low, high)
# All 30 runs are idempotent — re-run safely after interruption.

total = len(AEIOU_RIGIDITIES) * len(SPLIT_CONFIGS)
done  = 0

for r in AEIOU_RIGIDITIES:
    loss_name = f"aeiou_r{_fmt_r(r)}"
    loss_fn   = AEIOU_LOSS_REGISTRY[loss_name]

    for split_name, cfg_path in SPLIT_CONFIGS.items():
        done += 1
        print(f"\n[{done}/{total}] λ={r}  split={split_name}")
        run_training(
            loss_name=loss_name,
            loss_fn=loss_fn,
            split_name=split_name,
            yaml_path=cfg_path,
        )

restore_loss()  # Defensive final restore
print(f"\nAll {total} AEIoU grid runs complete (or skipped).")


## Section 9 · Results Collection

Ultralytics writes a `results.csv` file into each run directory after training.
This section collects all 33 CSVs into a single flat DataFrame tagged with
`loss`, `split`, and `rigidity` columns, then caches it to
`experiments_kvasir/all_results_combined.csv`.

**All downstream analysis cells load from this cache** — they never trigger
re-training. This means you can re-run Sections 10–13 on a fresh Colab session
(after uploading the cache CSV) without re-training any models.

### Column stripping
Ultralytics prefixes column names with a space (` metrics/mAP50-95(B)`).
We strip all column names with `df.columns.str.strip()` before any analysis.

**What to look for:** The combined DataFrame should have:
- 33 unique `run_name` values (3 EIoU + 30 AEIoU)
- 20 rows per run (one per epoch) → 660 rows total
- No NaN in `metrics/mAP50-95(B)` column at the final epoch


In [ ]:
# --- Load all results.csv files into a single flat DataFrame
import pandas as pd

CACHE_CSV = EXPERIMENTS / "all_results_combined.csv"

def load_all_results(force_rebuild=False):
    """
    Returns a DataFrame with all training metrics tagged by loss and split.
    Loads from cache if available; rebuilds from individual CSVs if not.
    """
    if CACHE_CSV.exists() and not force_rebuild:
        print(f"Loading from cache: {CACHE_CSV}")
        return pd.read_csv(CACHE_CSV)

    print("Building combined results from individual CSVs...")
    dfs = []

    # EIoU runs
    for split_name in SPLIT_CONFIGS:
        run_name = f"kvasir_yolo26n_eiou_{split_name}_e{EPOCHS}"
        csv_path = EXPERIMENTS / run_name / "results.csv"
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            df.columns = df.columns.str.strip()  # Remove Ultralytics leading spaces
            df["run_name"] = run_name
            df["loss"]     = "eiou"
            df["split"]    = split_name
            df["rigidity"] = -1.0   # Sentinel for EIoU (no λ)
            dfs.append(df)
        else:
            print(f"  [MISSING] {csv_path}")

    # AEIoU grid runs
    for r in AEIOU_RIGIDITIES:
        loss_name = f"aeiou_r{_fmt_r(r)}"
        for split_name in SPLIT_CONFIGS:
            run_name = f"kvasir_yolo26n_{loss_name}_{split_name}_e{EPOCHS}"
            csv_path = EXPERIMENTS / run_name / "results.csv"
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                df.columns = df.columns.str.strip()
                df["run_name"] = run_name
                df["loss"]     = loss_name
                df["split"]    = split_name
                df["rigidity"] = r
                dfs.append(df)
            else:
                print(f"  [MISSING] {csv_path}")

    if not dfs:
        raise RuntimeError("No results found. Run training cells first.")

    combined = pd.concat(dfs, ignore_index=True)
    combined.to_csv(CACHE_CSV, index=False)
    print(f"Saved combined CSV: {CACHE_CSV}  ({len(combined)} rows)")
    return combined


df_all = load_all_results()
print(f"\nCombined DataFrame shape: {df_all.shape}")
print(f"Unique runs: {df_all['run_name'].nunique()}")
print(f"Epochs per run: {df_all.groupby('run_name').size().unique()}")
print(f"\nColumns: {list(df_all.columns[:10])}...")


In [ ]:
# --- Final-epoch pivot table: mAP50-95 for each loss × split
# Shows at a glance which loss wins on which split condition.
# The 'best' cell per column is the AEIoU λ to recommend for that noise level.
import pandas as pd

MAP_COL = "metrics/mAP50-95(B)"

# Take the last epoch row for each run (epoch with highest index = final epoch)
df_final = (
    df_all.sort_values("epoch")
          .groupby("run_name")
          .last()
          .reset_index()
)

# Build pivot: rows = loss name, cols = split
pivot = df_final.pivot_table(index="loss", columns="split", values=MAP_COL)
# Order columns consistently
pivot = pivot[["clean", "low", "high"]]
# Add robustness ratio column: mAP_high / mAP_clean (higher = more robust)
pivot["robust_ratio"] = pivot["high"] / pivot["clean"]
# Sort by clean mAP descending
pivot = pivot.sort_values("clean", ascending=False)

# Save to CSV for the paper
pivot.to_csv(ANALYSIS_DIR / "summary_table.csv")

# Pretty-print with highlighting
print(f"{'Loss':<18} {'clean':>8} {'low':>8} {'high':>8} {'ratio':>8}")
print("-" * 55)
for loss_name, row in pivot.iterrows():
    marker = " ←best" if loss_name == pivot["clean"].idxmax() else ""
    print(f"{loss_name:<18} {row['clean']:>8.4f} {row['low']:>8.4f} "
          f"{row['high']:>8.4f} {row['robust_ratio']:>8.4f}{marker}")

print(f"\nBest clean mAP: {pivot['clean'].idxmax()} = {pivot['clean'].max():.4f}")
print(f"Best robust ratio: {pivot['robust_ratio'].idxmax()} = {pivot['robust_ratio'].max():.4f}")
print(f"Saved → {ANALYSIS_DIR / 'summary_table.csv'}")


## Section 10 · Bounding Box Visualisation on Real Polyp Images

We select **5 held-out validation images** spanning the polyp shape distribution
(small, large, elongated, round, flat) and run inference from 5 models:
- EIoU (baseline)
- AEIoU λ=0.1 (minimal shape penalty)
- AEIoU λ=0.3 (expected best)
- AEIoU λ=0.5 (moderate)
- AEIoU λ=1.0 (EIoU-equivalent rigidity)

For each prediction we compute IoU with the ground-truth box and print it on the
predicted box. This makes quality differences immediately visible.

**What to look for:**
- AEIoU(λ=0.3) boxes should show higher IoU scores on irregularly shaped polyps
- EIoU may over-correct box size on small polyps (enclosing-box normaliser effect)
- AEIoU(λ=0.1) may produce looser boxes (low shape penalty) but still good centers
- All models should detect the polyp; differences are in box tightness and IoU score


In [ ]:
# --- Select 5 stratified demo images from the validation set
# Stratified by GT box area to cover: tiny, small, medium, large, elongated polyps.
import cv2, numpy as np
from pathlib import Path

def select_kvasir_demo_images(img_dir, lbl_dir, n=5):
    """
    Select n validation images stratified by polyp bounding-box area.
    Returns list of (img_path, gt_box_xyxy) tuples for rendering.
    """
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)

    records = []
    for img_path in sorted(img_dir.glob("*.jpg")):
        lbl_path = lbl_dir / f"{img_path.stem}.txt"
        if not lbl_path.exists():
            continue
        line = lbl_path.read_text().strip().split("\n")[0]
        _, cx, cy, bw, bh = [float(v) for v in line.split()]
        # Load image to get dimensions
        img = cv2.imread(str(img_path))
        H, W = img.shape[:2]
        # Convert normalised xywh → pixel xyxy
        x1 = (cx - bw/2) * W;  y1 = (cy - bh/2) * H
        x2 = (cx + bw/2) * W;  y2 = (cy + bh/2) * H
        area = (x2 - x1) * (y2 - y1)
        ar   = (x2 - x1) / max((y2 - y1), 1)  # aspect ratio
        records.append((img_path, (x1,y1,x2,y2), area, ar))

    # Sort by area; pick n evenly spaced images across the area range
    records.sort(key=lambda r: r[2])
    indices = np.linspace(0, len(records)-1, n, dtype=int)
    selected = [records[i] for i in indices]
    print(f"Selected {len(selected)} demo images:")
    for p, box, area, ar in selected:
        print(f"  {p.name}: area={area:.0f}px²  AR={ar:.2f}")
    return selected


val_img_dir = DATASET_ROOT / "valid" / "images"
val_lbl_dir = DATASET_ROOT / "valid" / "labels"
DEMO_IMAGES = select_kvasir_demo_images(val_img_dir, val_lbl_dir, n=5)


In [ ]:
# --- Run inference from 5 models on demo images; cache results
# Caches to inference_cache.pkl so this cell only runs once even if
# downstream visualisation parameters change.
import pickle
from ultralytics import YOLO
from pathlib import Path

INFERENCE_CACHE = EXPERIMENTS / "inference_cache.pkl"

LOSSES_TO_VIS = ["eiou", "aeiou_r0p1", "aeiou_r0p3", "aeiou_r0p5", "aeiou_r1p0"]
# Use clean-split models for visualisation (best weights, no noise)
VIS_SPLIT = "clean"

if INFERENCE_CACHE.exists():
    print(f"Loading inference cache: {INFERENCE_CACHE}")
    with open(INFERENCE_CACHE, "rb") as f:
        inference_results = pickle.load(f)
else:
    print("Running inference on demo images...")
    inference_results = {}

    for loss_name in LOSSES_TO_VIS:
        run_name = f"kvasir_yolo26n_{loss_name}_{VIS_SPLIT}_e{EPOCHS}"
        weights  = EXPERIMENTS / run_name / "weights" / "best.pt"

        if not weights.exists():
            print(f"  [SKIP] {run_name} weights not found")
            continue

        print(f"  Loading {run_name}...")
        model = YOLO(str(weights))

        preds = {}
        for img_path, gt_box, area, ar in DEMO_IMAGES:
            result = model(str(img_path), verbose=False)[0]
            # Store boxes in xyxy format (pixel coords) and confidence scores
            boxes  = result.boxes.xyxy.cpu().numpy() if result.boxes else np.zeros((0,4))
            scores = result.boxes.conf.cpu().numpy() if result.boxes else np.zeros(0)
            preds[img_path.name] = {"boxes": boxes, "scores": scores}
        inference_results[loss_name] = preds

    with open(INFERENCE_CACHE, "wb") as f:
        pickle.dump(inference_results, f)
    print(f"Cached → {INFERENCE_CACHE}")

print(f"Inference results loaded for: {list(inference_results.keys())}")


In [ ]:
# --- Draw bbox comparison grid: 5 images × 5 models
# For each predicted box, computes IoU with GT box and prints it as a label.
# Ground truth shown as white dashed box; predictions in model-specific colors.
import cv2, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches


def _compute_iou(boxA, boxB):
    """Compute IoU between two xyxy boxes (numpy arrays)."""
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB-xA) * max(0, yB-yA)
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter + 1e-7)


MODEL_COLORS = {
    "eiou":       "#E63946",
    "aeiou_r0p1": "#457B9D",
    "aeiou_r0p3": "#2A9D8F",
    "aeiou_r0p5": "#F4A261",
    "aeiou_r1p0": "#6A4C93",
}
LOSS_LABELS = {
    "eiou":       "EIoU",
    "aeiou_r0p1": "AEIoU λ=0.1",
    "aeiou_r0p3": "AEIoU λ=0.3",
    "aeiou_r0p5": "AEIoU λ=0.5",
    "aeiou_r1p0": "AEIoU λ=1.0",
}

n_rows = len(DEMO_IMAGES)
n_cols = len(LOSSES_TO_VIS)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
fig.suptitle("Kvasir-SEG: Predicted boxes — EIoU vs AEIoU λ grid\n"
             "(white dashed = GT, colored solid = prediction, label = IoU with GT)",
             fontsize=12, fontweight="bold", y=1.01)

for row, (img_path, gt_box, area, ar) in enumerate(DEMO_IMAGES):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]
    gt_np = np.array(gt_box)  # xyxy pixel

    for col, loss_name in enumerate(LOSSES_TO_VIS):
        ax = axes[row, col]
        ax.imshow(img)

        # Ground truth box — white dashed
        x1,y1,x2,y2 = gt_np
        rect_gt = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                     linewidth=1.5, edgecolor="white",
                                     facecolor="none", linestyle="--")
        ax.add_patch(rect_gt)

        # Predicted boxes
        color = MODEL_COLORS.get(loss_name, "red")
        preds = inference_results.get(loss_name, {}).get(img_path.name, {})
        boxes  = preds.get("boxes",  np.zeros((0,4)))
        scores = preds.get("scores", np.zeros(0))

        for bi, (pb, sc) in enumerate(zip(boxes, scores)):
            iou = _compute_iou(pb, gt_np)
            rx1,ry1,rx2,ry2 = pb
            rect = patches.Rectangle((rx1,ry1), rx2-rx1, ry2-ry1,
                                      linewidth=2, edgecolor=color, facecolor="none")
            ax.add_patch(rect)
            ax.text(rx1, ry1-5, f"IoU={iou:.2f}", color=color, fontsize=7,
                    fontweight="bold",
                    bbox=dict(facecolor="black", alpha=0.6, pad=1))

        # Column header (top row only)
        if row == 0:
            ax.set_title(LOSS_LABELS.get(loss_name, loss_name), fontsize=9,
                         fontweight="bold", color=color)
        # Row label (leftmost column only)
        if col == 0:
            ax.set_ylabel(f"area={area:.0f}\nAR={ar:.2f}", fontsize=8)
        ax.axis("off")

plt.tight_layout()
save_path = ANALYSIS_DIR / "04_bbox_visual_comparison.png"
plt.savefig(save_path, dpi=120, bbox_inches="tight")  # → experiments_kvasir/analysis/04_bbox_visual_comparison.png
plt.show()
print(f"Saved → {save_path}")


**Figure 4 · Bounding Box Comparison Grid**

*Rows:* 5 validation images spanning the polyp size/shape range (small → large, round → elongated)
*Columns:* EIoU, AEIoU λ=0.1, 0.3, 0.5, 1.0 · *White dashed:* GT box · *Colored solid:* predicted box

**Expected pattern:**
- All models should detect the polyp (non-empty prediction box)
- AEIoU(λ=0.3) boxes should generally show higher IoU values on smaller, irregular polyps
- EIoU may over-fit box size on large polyps (enclosing-box normaliser lenient → larger predicted box)
- AEIoU(λ=0.1) may produce slightly loose boxes but well-centred
- On round, easily-labelled polyps (high AR≈1), differences should be minimal


In [ ]:
# --- Side-by-side zoom: EIoU vs best AEIoU for each demo image
# Identify best AEIoU λ from the pivot table (highest clean mAP)
best_aeiou = pivot[pivot.index != "eiou"]["clean"].idxmax()
print(f"Best AEIoU by clean mAP: {best_aeiou}")

fig, axes = plt.subplots(len(DEMO_IMAGES), 2,
                          figsize=(10, 4 * len(DEMO_IMAGES)))
fig.suptitle(f"EIoU vs {best_aeiou} — side-by-side on 5 demo images",
             fontsize=13, fontweight="bold")

for row, (img_path, gt_box, area, ar) in enumerate(DEMO_IMAGES):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    gt_np = np.array(gt_box)

    for col, loss_name in enumerate(["eiou", best_aeiou]):
        ax = axes[row, col]
        ax.imshow(img)

        # GT box
        x1,y1,x2,y2 = gt_np
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                        linewidth=1.5, edgecolor="white",
                                        facecolor="none", linestyle="--"))

        color = MODEL_COLORS.get(loss_name, "red")
        preds = inference_results.get(loss_name, {}).get(img_path.name, {})
        for pb in preds.get("boxes", []):
            iou = _compute_iou(pb, gt_np)
            rx1,ry1,rx2,ry2 = pb
            ax.add_patch(patches.Rectangle((rx1,ry1), rx2-rx1, ry2-ry1,
                                            linewidth=2.5, edgecolor=color, facecolor="none"))
            ax.text(rx1, ry1-5, f"polyp  IoU={iou:.2f}", color=color, fontsize=9,
                    fontweight="bold",
                    bbox=dict(facecolor="black", alpha=0.6, pad=1))

        label = LOSS_LABELS.get(loss_name, loss_name)
        ax.set_title(f"{label}  (area={area:.0f}, AR={ar:.2f})", fontsize=9)
        ax.axis("off")

plt.tight_layout()
save_path = ANALYSIS_DIR / "04b_side_by_side_zoom.png"
plt.savefig(save_path, dpi=120, bbox_inches="tight")  # → experiments_kvasir/analysis/04b_side_by_side_zoom.png
plt.show()
print(f"Saved → {save_path}")


**Figure 4b · EIoU vs Best AEIoU — Side-by-Side**

*Left column:* EIoU predictions · *Right column:* Best AEIoU (λ determined from pivot table)
*White dashed:* GT box · *Labels:* class name + IoU score

**Expected pattern:** The best AEIoU column should show equal or higher IoU scores,
especially on small (<10% image) and irregular (AR far from 1.0) polyps.
If IoU scores are identical on most images, the two losses may have converged to
equivalent solutions at this epoch count — consider running 50 epochs.


## Section 11 · Deep EIoU vs AEIoU Analysis

Seven quantitative analyses provide a multi-angle view of how EIoU and AEIoU differ:

| # | Analysis | Key question |
|---|---|---|
| 1 | Final mAP bar chart | Which loss achieves highest mAP per split? |
| 2 | Learning curves | Does AEIoU converge differently? |
| 3 | Convergence speed | Which loss reaches 90% of final mAP fastest? |
| 4 | Noise robustness gap | Which loss degrades least under label noise? |
| 5 | λ sensitivity heatmap | Which λ wins on which split? Is optimal λ stable? |
| 6 | mAP@thresholds curve | Do gains hold across all IoU thresholds? |
| 7 | Training stability | Does AEIoU produce smoother loss curves? |


In [ ]:
# --- (1) Final mAP bar chart: EIoU vs AEIoU λ grid per split
import matplotlib.pyplot as plt
import numpy as np

splits    = ["clean", "low", "high"]
split_colors = {"clean": "#4CAF50", "low": "#FFC107", "high": "#F44336"}
x_labels = ["eiou"] + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]
x_pos    = np.arange(len(x_labels))
width    = 0.25

fig, ax = plt.subplots(figsize=(14, 5))
for i, split in enumerate(splits):
    vals = [pivot.loc[l, split] if l in pivot.index else 0 for l in x_labels]
    bars = ax.bar(x_pos + i*width, vals, width=width,
                  color=split_colors[split], alpha=0.85, label=split)

# Mark EIoU position with vertical line
ax.axvline(x=0 + width, color="black", linestyle=":", lw=1.2, alpha=0.6)
ax.set_xticks(x_pos + width)
ax.set_xticklabels(
    ["EIoU"] + [f"λ={r}" for r in AEIOU_RIGIDITIES],
    rotation=40, ha="right", fontsize=9
)
ax.set_ylabel("mAP@[.5:.95]", fontsize=11)
ax.set_title("Final mAP: EIoU vs AEIoU λ grid across 3 noise splits", fontsize=12, fontweight="bold")
ax.legend(title="Split", fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "02_final_map_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/02_final_map_comparison.png
plt.show()
print(f"Saved → {save_path}")


**Figure 2 · Final mAP Comparison**

*x-axis:* Loss function (EIoU then AEIoU λ=0.1→1.0) · *y-axis:* mAP@[.5:.95]
*Green/yellow/red bars:* clean / low-noise / high-noise split

**Expected pattern:** AEIoU at some λ ∈ [0.2, 0.4] should exceed EIoU on the clean
split. The high-noise bars should degrade least for low λ values.
AEIoU(λ=1.0) bar should be close to but not identical to EIoU (different normaliser).


In [ ]:
# --- (2) Learning curves: EIoU vs best AEIoU on all 3 splits
import matplotlib.pyplot as plt

best_aeiou = pivot[pivot.index != "eiou"]["clean"].idxmax()
print(f"Best AEIoU (clean mAP): {best_aeiou}")

MAP_COL = "metrics/mAP50-95(B)"
LOSS_COL = "train/box_loss"

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f"Learning curves — EIoU vs {best_aeiou}", fontsize=13, fontweight="bold")

for col_idx, split in enumerate(["clean", "low", "high"]):
    for row_idx, metric in enumerate([LOSS_COL, MAP_COL]):
        ax = axes[row_idx, col_idx]

        for loss_name, color, label in [
            ("eiou",      "#E63946", "EIoU"),
            (best_aeiou,  "#2A9D8F", best_aeiou),
        ]:
            run_name = f"kvasir_yolo26n_{loss_name}_{split}_e{EPOCHS}"
            sub = df_all[df_all["run_name"] == run_name].sort_values("epoch")
            if sub.empty:
                continue
            if metric in sub.columns:
                ax.plot(sub["epoch"], sub[metric], color=color, lw=2, label=label)

        ax.set_title(f"{metric.split('/')[1]}  —  {split}", fontsize=9)
        ax.set_xlabel("Epoch")
        if col_idx == 0:
            ax.set_ylabel(metric.split("/")[1])
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.tight_layout()
save_path = ANALYSIS_DIR / "05_learning_curves.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/05_learning_curves.png
plt.show()
print(f"Saved → {save_path}")


**Figure 5 · Learning Curves**

*Top row:* training box loss per epoch · *Bottom row:* validation mAP per epoch
*Columns:* clean / low-noise / high-noise splits · *Red:* EIoU · *Teal:* best AEIoU

**Expected pattern:** AEIoU may reach a comparable or higher final mAP in fewer epochs
(faster convergence). The box loss curves should be smoother for AEIoU at low λ because
the size penalty is reduced, making the gradient signal less noisy.


In [ ]:
# --- (3) Convergence speed: epoch at which each model first reaches 90% of final mAP
import matplotlib.pyplot as plt
import numpy as np

MAP_COL   = "metrics/mAP50-95(B)"
threshold = 0.90  # 90% of final mAP

all_losses = ["eiou"] + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]
conv_data  = {split: [] for split in ["clean", "low", "high"]}

for loss_name in all_losses:
    for split in ["clean", "low", "high"]:
        run_name = f"kvasir_yolo26n_{loss_name}_{split}_e{EPOCHS}"
        sub = df_all[df_all["run_name"] == run_name].sort_values("epoch")
        if sub.empty or MAP_COL not in sub.columns:
            conv_data[split].append(EPOCHS)
            continue
        final_map = sub[MAP_COL].iloc[-1]
        target    = threshold * final_map
        # Find first epoch where mAP >= 90% of final value
        reached = sub[sub[MAP_COL] >= target]["epoch"]
        conv_epoch = int(reached.iloc[0]) if len(reached) else EPOCHS
        conv_data[split].append(conv_epoch)

x_pos   = np.arange(len(all_losses))
x_labels = ["EIoU"] + [f"λ={r}" for r in AEIOU_RIGIDITIES]
width   = 0.25
fig, ax = plt.subplots(figsize=(13, 5))
for i, (split, color) in enumerate(zip(["clean","low","high"],
                                        ["#4CAF50","#FFC107","#F44336"])):
    ax.bar(x_pos + i*width, conv_data[split], width=width, color=color, alpha=0.85, label=split)

ax.set_xticks(x_pos + width)
ax.set_xticklabels(x_labels, rotation=40, ha="right", fontsize=9)
ax.set_ylabel(f"Epoch to reach {threshold*100:.0f}% of final mAP", fontsize=11)
ax.set_title("Convergence speed: epochs to 90% final mAP", fontsize=12, fontweight="bold")
ax.legend(title="Split")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "06_convergence_speed.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/06_convergence_speed.png
plt.show()
print(f"Saved → {save_path}")


**Figure 6 · Convergence Speed**

*x-axis:* Loss function · *y-axis:* Epoch at which model first reaches 90% of its final mAP
*Lower bar = faster convergence*

**Expected pattern:** Low-λ AEIoU may converge faster because the simplified loss
landscape (no strong shape penalty) is easier for the optimiser to navigate in early
epochs. If all bars are equal, 20 epochs is too short to see convergence differences.


In [ ]:
# --- (4) Noise robustness gap: mAP_clean - mAP_high for each loss
# Smaller gap = more robust to annotation noise
import matplotlib.pyplot as plt
import numpy as np

all_losses = ["eiou"] + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]
gaps = []
for loss_name in all_losses:
    clean_map = pivot.loc[loss_name, "clean"] if loss_name in pivot.index else 0
    high_map  = pivot.loc[loss_name, "high"]  if loss_name in pivot.index else 0
    gaps.append(clean_map - high_map)

x_labels = ["EIoU"] + [f"λ={r}" for r in AEIOU_RIGIDITIES]
colors    = ["#E63946"] + [PALETTE.get(f"aeiou_r{_fmt_r(r)}", "#888") for r in AEIOU_RIGIDITIES]
# Use a gradient for AEIoU bars
import matplotlib.cm as cm
aeiou_colors = [cm.Blues(0.3 + 0.07*i) for i in range(len(AEIOU_RIGIDITIES))]
bar_colors   = ["#E63946"] + [f"#{int(c[0]*255):02x}{int(c[1]*255):02x}{int(c[2]*255):02x}"
                               for c in aeiou_colors]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(x_labels, gaps, color=bar_colors, edgecolor="white", lw=0.5)
ax.axhline(y=gaps[0], color="#E63946", linestyle="--", lw=1.5,
           label=f"EIoU gap = {gaps[0]:.4f}")
ax.set_ylabel("mAP gap (clean − high)", fontsize=11)
ax.set_title("Noise robustness gap — smaller is better", fontsize=12, fontweight="bold")
ax.tick_params(axis="x", rotation=40)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
# Annotate bars with gap values
for bar, gap in zip(bars, gaps):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{gap:.4f}", ha="center", va="bottom", fontsize=7)
plt.tight_layout()
save_path = ANALYSIS_DIR / "07_noise_robustness_gap.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/07_noise_robustness_gap.png
plt.show()

# Print ranking
print("\nNoise robustness ranking (smaller gap = more robust):")
ranked = sorted(zip(x_labels, gaps), key=lambda x: x[1])
for rank, (name, gap) in enumerate(ranked, 1):
    print(f"  {rank:2d}. {name:<18} gap={gap:.4f}")


**Figure 7 · Noise Robustness Gap**

*x-axis:* Loss function · *y-axis:* mAP_clean − mAP_high (smaller = more robust)
*Red dashed line:* EIoU gap (baseline)

**Expected pattern:** Low-λ AEIoU values (0.1–0.3) should have smaller gaps than EIoU,
confirming that down-weighting the shape penalty makes the model less sensitive to
annotation noise. AEIoU(λ=1.0) should have a gap similar to EIoU.


In [ ]:
# --- (5) Lambda sensitivity heatmap
# Key figure: shows optimal λ per split and whether it's stable across conditions.
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

heat_data = []
for r in AEIOU_RIGIDITIES:
    loss_name = f"aeiou_r{_fmt_r(r)}"
    row = {"lambda": r}
    for split in ["clean", "low", "high"]:
        val = pivot.loc[loss_name, split] if loss_name in pivot.index else float("nan")
        row[split] = val
    heat_data.append(row)

heat_df = pd.DataFrame(heat_data).set_index("lambda")

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heat_df.T, annot=True, fmt=".4f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "mAP@[.5:.95]"})
ax.set_title("AEIoU λ sensitivity heatmap\n(rows=split, cols=λ)", fontsize=12, fontweight="bold")
ax.set_xlabel("Rigidity λ", fontsize=11)
ax.set_ylabel("Split", fontsize=11)
plt.tight_layout()
save_path = ANALYSIS_DIR / "03_aeiou_rigidity_heatmap.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/03_aeiou_rigidity_heatmap.png
plt.show()

# Report optimal λ per split
for split in ["clean", "low", "high"]:
    best_lambda = heat_df[split].idxmax()
    best_val    = heat_df[split].max()
    print(f"  Best λ for {split}: {best_lambda}  (mAP={best_val:.4f})")
print(f"\nSaved → {save_path}")


**Figure 3 · AEIoU λ Sensitivity Heatmap**

*Rows:* split conditions · *Columns:* λ values (0.1 → 1.0)
*Cell values:* mAP@[.5:.95] · *Colour:* darker = higher mAP

**Expected pattern:** Peak mAP should appear at λ ∈ [0.2, 0.4] across all splits,
confirming the optimal λ is stable. If the peak shifts significantly between clean
(highest λ) and high-noise (lowest λ), that confirms AEIoU adapts the shape penalty
correctly to noise level. If all columns are equal, λ has no effect — that would
suggest the size term is numerically negligible.


In [ ]:
# --- (6) mAP@[0.5:0.95] threshold curve
# Computes AP at each IoU threshold for EIoU vs best AEIoU to show that
# improvements are consistent across all thresholds, not just the average.
# Uses model.val() on the clean split.
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

best_aeiou = pivot[pivot.index != "eiou"]["clean"].idxmax()
THRESHOLD_RANGE = np.arange(0.50, 1.00, 0.05)

results_by_thresh = {}
for loss_name in ["eiou", best_aeiou]:
    run_name = f"kvasir_yolo26n_{loss_name}_clean_e{EPOCHS}"
    weights  = EXPERIMENTS / run_name / "weights" / "best.pt"
    if not weights.exists():
        print(f"[SKIP] {run_name} weights not found")
        continue

    model   = YOLO(str(weights))
    aps     = []
    for thresh in THRESHOLD_RANGE:
        val_res = model.val(
            data=str(SPLIT_CONFIGS["clean"]),
            iou=float(thresh), verbose=False,
        )
        aps.append(float(val_res.box.ap50 if hasattr(val_res.box, "ap50")
                         else val_res.box.maps[0]))
    results_by_thresh[loss_name] = aps
    print(f"  {loss_name}: {aps}")

fig, ax = plt.subplots(figsize=(8, 5))
for loss_name, color in [("eiou", "#E63946"), (best_aeiou, "#2A9D8F")]:
    if loss_name in results_by_thresh:
        ax.plot(THRESHOLD_RANGE, results_by_thresh[loss_name],
                marker="o", color=color, lw=2,
                label=LOSS_LABELS.get(loss_name, loss_name))

ax.set_xlabel("IoU threshold", fontsize=11)
ax.set_ylabel("AP", fontsize=11)
ax.set_title("AP vs IoU threshold (clean split)", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "09_map_at_thresholds.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/09_map_at_thresholds.png
plt.show()
print(f"Saved → {save_path}")


**Figure 9 · AP vs IoU Threshold**

*x-axis:* IoU threshold (0.5 → 0.95) · *y-axis:* Average Precision at that threshold
*Red:* EIoU · *Teal:* best AEIoU

**Expected pattern:** Best AEIoU curve should stay at or above EIoU across all thresholds.
If AEIoU is better only at low thresholds (IoU=0.5) but worse at strict thresholds
(IoU=0.9), the boxes are better-centred but not tighter — consistent with low-λ AEIoU
deprioritising exact size matching.


In [ ]:
# --- (7) Training loss stability: rolling std of box_loss
# Lower variance = more stable gradient signal = easier optimisation
import matplotlib.pyplot as plt

LOSS_COL = "train/box_loss"
WINDOW   = 3  # epochs for rolling std

best_aeiou = pivot[pivot.index != "eiou"]["clean"].idxmax()
fig, axes  = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Training box loss stability (rolling std, window=3 epochs)",
             fontsize=12, fontweight="bold")

for ax, split in zip(axes, ["clean", "high"]):
    for loss_name, color, label in [
        ("eiou",     "#E63946", "EIoU"),
        (best_aeiou, "#2A9D8F", best_aeiou),
    ]:
        run_name = f"kvasir_yolo26n_{loss_name}_{split}_e{EPOCHS}"
        sub = df_all[df_all["run_name"] == run_name].sort_values("epoch")
        if sub.empty or LOSS_COL not in sub.columns:
            continue
        loss_vals = sub[LOSS_COL].values
        rolling_std = pd.Series(loss_vals).rolling(WINDOW, min_periods=1).std().values
        epochs_arr  = sub["epoch"].values
        ax.plot(epochs_arr, loss_vals, color=color, lw=2, label=label)
        ax.fill_between(epochs_arr,
                        loss_vals - rolling_std, loss_vals + rolling_std,
                        color=color, alpha=0.15)
    ax.set_title(f"Split: {split}", fontsize=10)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("box_loss")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
save_path = ANALYSIS_DIR / "11_training_stability.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/11_training_stability.png
plt.show()
print(f"Saved → {save_path}")


**Figure 11 · Training Loss Stability**

*x-axis:* Epoch · *y-axis:* Training box loss · *Shaded region:* ± rolling std (window=3)
*Left:* clean split · *Right:* high-noise split

**Expected pattern:** AEIoU (teal) should have a narrower shaded band (lower variance)
than EIoU (red), particularly on the high-noise split. If AEIoU(low λ) is more stable,
it confirms that the down-weighted shape penalty reduces gradient noise from imprecise labels.


## Section 12 · Validity Checks

Three additional checks provide evidence that AEIoU improvements are real and not
due to random variation or measurement artefacts.

| Check | Null hypothesis | Evidence against null |
|---|---|---|
| Box IoU distribution | AEIoU and EIoU produce same box quality | Right-shifted KDE for AEIoU |
| Polyp-size stratified AP | Both losses equally affect all polyp sizes | AEIoU gains concentrated on small/irregular polyps |
| Noise robustness ranking | All losses equally robust | AEIoU low-λ ranks #1 in mAP_high/mAP_clean |


In [ ]:
# --- Box IoU distribution: KDE of per-prediction IoU scores
# For each demo image, collects IoU(prediction, GT) for EIoU and best AEIoU.
# A right-shifted distribution for AEIoU = systematically higher box quality.
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

best_aeiou = pivot[pivot.index != "eiou"]["clean"].idxmax()
all_ious   = {"eiou": [], best_aeiou: []}

for loss_name in all_ious.keys():
    preds_for_loss = inference_results.get(loss_name, {})
    for img_path, gt_box, area, ar in DEMO_IMAGES:
        gt_np = np.array(gt_box)
        preds = preds_for_loss.get(img_path.name, {})
        for pb in preds.get("boxes", []):
            iou = _compute_iou(pb, gt_np)
            all_ious[loss_name].append(iou)

fig, ax = plt.subplots(figsize=(8, 4))
x_range = np.linspace(0, 1, 200)
for loss_name, color, label in [
    ("eiou",     "#E63946", "EIoU"),
    (best_aeiou, "#2A9D8F", best_aeiou),
]:
    data = all_ious[loss_name]
    if len(data) < 2:
        continue
    kde = gaussian_kde(data, bw_method=0.2)
    ax.plot(x_range, kde(x_range), color=color, lw=2.5, label=label)
    ax.fill_between(x_range, kde(x_range), color=color, alpha=0.12)
    ax.axvline(np.mean(data), color=color, linestyle="--", lw=1.2,
               label=f"{label} mean={np.mean(data):.3f}")

ax.set_xlabel("IoU with ground truth box", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Per-prediction IoU distribution on demo images", fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "10_box_iou_distribution.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/10_box_iou_distribution.png
plt.show()
print(f"Saved → {save_path}")
print(f"EIoU mean IoU:      {np.mean(all_ious['eiou']):.4f}")
print(f"{best_aeiou} mean IoU: {np.mean(all_ious[best_aeiou]):.4f}")


**Figure 10 · Per-Prediction IoU Distribution**

*x-axis:* IoU between predicted box and GT box (higher = better prediction quality)
*y-axis:* Probability density (KDE) · *Dashed vertical:* mean IoU

**Expected pattern:** AEIoU curve (teal) should be shifted right (higher IoU values)
compared to EIoU (red). If means are within 0.01 of each other, the effect is small
but the distribution shape may still differ (e.g. AEIoU fewer very-low IoU outliers).


In [ ]:
# --- Polyp-size stratified AP
# Split valid images into 3 tertiles by GT box area.
# Compute AP50 for EIoU vs best AEIoU separately for each size stratum.
# Expected: AEIoU gains concentrated on small polyps (hardest, most irregular).
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
from pathlib import Path

best_aeiou = pivot[pivot.index != "eiou"]["clean"].idxmax()
val_lbl_dir = DATASET_ROOT / "valid" / "labels"
val_img_dir = DATASET_ROOT / "valid" / "images"

# Compute GT box areas for all valid images
areas = []
for lbl_path in sorted(val_lbl_dir.glob("*.txt")):
    line = lbl_path.read_text().strip().split("\n")[0]
    _, cx, cy, bw, bh = [float(v) for v in line.split()]
    # Use normalised area (bw * bh) — invariant to image resolution
    areas.append((lbl_path.stem, bw * bh))

areas.sort(key=lambda x: x[1])
n = len(areas)
# Three tertiles: small / medium / large
strata = {
    "small":  [s for s, _ in areas[:n//3]],
    "medium": [s for s, _ in areas[n//3:2*n//3]],
    "large":  [s for s, _ in areas[2*n//3:]],
}
print(f"Size strata: small={len(strata['small'])} "
      f"medium={len(strata['medium'])} large={len(strata['large'])}")

# Compute mean max-IoU per stratum as a proxy for AP50
# (Full AP computation requires running model.val() with per-image filtering)
stratified_results = {"small": {}, "medium": {}, "large": {}}

for loss_name in ["eiou", best_aeiou]:
    run_name = f"kvasir_yolo26n_{loss_name}_clean_e{EPOCHS}"
    weights  = EXPERIMENTS / run_name / "weights" / "best.pt"
    if not weights.exists():
        print(f"[SKIP] {run_name}")
        continue
    model = YOLO(str(weights))

    for stratum, stems in strata.items():
        iou_list = []
        for stem in stems:
            img_path = val_img_dir / f"{stem}.jpg"
            lbl_path = val_lbl_dir / f"{stem}.txt"
            if not img_path.exists():
                continue
            res = model(str(img_path), verbose=False)[0]
            # GT box
            line  = lbl_path.read_text().strip().split("\n")[0]
            _, cx, cy, bw, bh = [float(v) for v in line.split()]
            img   = plt.imread(str(img_path))
            H, W  = img.shape[:2]
            gt_np = np.array([(cx-bw/2)*W,(cy-bh/2)*H,(cx+bw/2)*W,(cy+bh/2)*H])
            if res.boxes and len(res.boxes.xyxy):
                pb  = res.boxes.xyxy.cpu().numpy()[0]
                iou = _compute_iou(pb, gt_np)
            else:
                iou = 0.0
            iou_list.append(iou)
        stratified_results[stratum][loss_name] = np.mean(iou_list) if iou_list else 0

print("\nStratified mean IoU results:")
for stratum in ["small","medium","large"]:
    for ln in ["eiou", best_aeiou]:
        print(f"  {stratum} | {ln}: {stratified_results[stratum].get(ln,0):.4f}")

# Bar chart
stratum_names = ["small", "medium", "large"]
x = np.arange(len(stratum_names))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
eiou_vals  = [stratified_results[s].get("eiou", 0)      for s in stratum_names]
aeiou_vals = [stratified_results[s].get(best_aeiou, 0)  for s in stratum_names]
ax.bar(x - w/2, eiou_vals,  w, color="#E63946", label="EIoU",        alpha=0.85)
ax.bar(x + w/2, aeiou_vals, w, color="#2A9D8F", label=best_aeiou,    alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(["Small polyps\n(bottom 33%)","Medium polyps\n(middle 33%)","Large polyps\n(top 33%)"])
ax.set_ylabel("Mean prediction IoU", fontsize=11)
ax.set_title("Polyp-size stratified prediction quality", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_path = ANALYSIS_DIR / "08_size_stratified_ap.png"
plt.savefig(save_path, dpi=150, bbox_inches="tight")  # → experiments_kvasir/analysis/08_size_stratified_ap.png
plt.show()
print(f"Saved → {save_path}")


**Figure 8 · Polyp-Size Stratified Prediction Quality**

*x-axis:* Polyp size stratum (by GT box area) · *y-axis:* Mean IoU between prediction and GT
*Red:* EIoU · *Teal:* best AEIoU

**Expected pattern:** AEIoU should gain most on **small polyps** (left bar).
Small polyps have highly irregular boundaries relative to their size, making the
EIoU enclosing-box normaliser overly lenient (large enclosing box relative to polyp).
AEIoU's target-dim normaliser scales the penalty to the polyp itself.
On **large polyps** (right bar), differences should be smaller — large polyps are
easier to detect and box size errors are proportionally smaller.


In [ ]:
# --- Noise robustness ranking table
# Rank all 11 losses (EIoU + 10 AEIoU) by mAP_high / mAP_clean ratio.
# Higher ratio = smaller degradation under severe noise = more robust.
import pandas as pd

all_losses_list = ["eiou"] + [f"aeiou_r{_fmt_r(r)}" for r in AEIOU_RIGIDITIES]

ranking = []
for loss_name in all_losses_list:
    if loss_name not in pivot.index:
        continue
    clean_map = pivot.loc[loss_name, "clean"]
    low_map   = pivot.loc[loss_name, "low"]
    high_map  = pivot.loc[loss_name, "high"]
    ratio     = high_map / clean_map if clean_map > 0 else 0
    rigidity  = -1 if loss_name == "eiou" else float(loss_name.split("_r")[1].replace("p","."))
    ranking.append({
        "loss":         loss_name,
        "rigidity":     rigidity,
        "clean_map":    clean_map,
        "low_map":      low_map,
        "high_map":     high_map,
        "robust_ratio": ratio,
    })

rank_df = pd.DataFrame(ranking).sort_values("robust_ratio", ascending=False).reset_index(drop=True)
rank_df.index += 1  # 1-based ranking

print("Noise robustness ranking (mAP_high / mAP_clean — higher = more robust):")
print(rank_df.to_string())

save_path = ANALYSIS_DIR / "robustness_ranking.csv"
rank_df.to_csv(save_path)
print(f"\nSaved → {save_path}")

# Report champion
champion = rank_df.iloc[0]
print(f"\n★ Most robust model: {champion['loss']}  "
      f"ratio={champion['robust_ratio']:.4f}  "
      f"clean={champion['clean_map']:.4f}  high={champion['high_map']:.4f}")


## Section 13 · Summary & Conclusions

### Key Findings

| Analysis | Finding | Supports AEIoU? |
|---|---|---|
| Final mAP (Fig 2) | AEIoU(λ=best) ≥ EIoU on clean split | ✓ if true |
| λ heatmap (Fig 3) | Optimal λ ∈ [0.2, 0.4] across splits | ✓ stable λ |
| Bbox visual (Fig 4) | AEIoU boxes show higher IoU on small polyps | ✓ qualitative |
| Learning curves (Fig 5) | AEIoU converges ≤ EIoU epochs | ✓ if faster |
| Convergence speed (Fig 6) | AEIoU reaches 90% mAP in fewer epochs | ✓ if lower bar |
| Robustness gap (Fig 7) | AEIoU gap < EIoU gap on high-noise split | ✓ key result |
| mAP@thresholds (Fig 9) | AEIoU ≥ EIoU across all IoU thresholds | ✓ consistent |
| IoU distribution (Fig 10) | AEIoU mean IoU > EIoU mean IoU | ✓ box quality |
| Size stratified (Fig 8) | AEIoU gains largest on small polyps | ✓ targeted |
| Robustness ranking (CSV) | AEIoU low-λ ranks #1 | ✓ strongest argument |

### Cross-dataset validation

If the optimal λ found here on Kvasir-SEG matches the optimal λ from notebook 02
(DUO underwater dataset), this is strong evidence that **λ ≈ 0.3 is a domain-agnostic
default** for amorphous object detection — no dataset-specific tuning required.

### Recommended settings for Kvasir-SEG polyp detection
```
loss = AEIoULoss(rigidity=0.3, reduction="none")
```

### Future work
- Increase epochs to 50–100 for clearer convergence comparisons
- Test on a larger model (yolo26s, yolo26m) — nano may have too little capacity
- Try on CVC-ClinicDB and ETIS-LaribPolypDB for further cross-dataset validation
- Investigate λ scheduling: start with λ=0.1 (explore centers) → anneal to λ=0.5
- Explore AEIoU for other amorphous medical tasks: skin lesion detection, cell nuclei


In [ ]:
# --- Save all remaining artifacts and print manifest
import json, os

print("=== Experiment Artifact Summary ===\n")
print(f"Experiments dir: {EXPERIMENTS}")
print(f"Analysis dir:    {ANALYSIS_DIR}")
print()

# List all saved figures
print("Saved figures:")
for f in sorted(ANALYSIS_DIR.glob("*.png")):
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:<45} {size_kb:>6.1f} KB")

# List saved CSVs
print("\nSaved tables:")
for f in sorted(ANALYSIS_DIR.glob("*.csv")):
    print(f"  {f.name}")

# Print manifest summary
if MANIFEST_PATH.exists():
    manifest = json.loads(MANIFEST_PATH.read_text())
    complete = sum(1 for v in manifest.values() if v.get("status") == "complete")
    failed   = sum(1 for v in manifest.values() if v.get("status") == "failed")
    print(f"\nManifest: {len(manifest)} runs  |  {complete} complete  |  {failed} failed")
    if failed:
        print("  Failed runs:")
        for k,v in manifest.items():
            if v.get("status") == "failed":
                print(f"    {k}: {v.get('error','?')}")

print("\nAll artifacts saved successfully.")


In [ ]:
# --- Final sync: push analysis figures and manifest to Drive
# Individual run directories were already synced after each training run (in finally block).
# This final cell ensures the analysis figures and summary CSVs are also saved to Drive.
import shutil

if DRIVE_AVAILABLE:
    # Sync analysis figures directory
    drive_analysis = DRIVE_EXPERIMENTS / "analysis"
    drive_analysis.mkdir(parents=True, exist_ok=True)
    if ANALYSIS_DIR.exists():
        shutil.copytree(str(ANALYSIS_DIR), str(drive_analysis), dirs_exist_ok=True)
        n_figs = len(list(drive_analysis.glob("*.png")))
        print(f"Analysis figures synced to Drive: {n_figs} PNGs")

    # Sync manifest and combined results CSV
    for fname in ["manifest.json", "all_results_combined.csv"]:
        src = EXPERIMENTS / fname
        if src.exists():
            shutil.copy2(str(src), str(DRIVE_EXPERIMENTS / fname))
            print(f"  Synced {fname}")

    print(f"\nAll artifacts backed up to: {DRIVE_EXPERIMENTS}")
    n_total = sum(1 for _ in DRIVE_EXPERIMENTS.rglob("*") if _.is_file())
    print(f"Total files on Drive: {n_total}")
else:
    print("Drive not mounted — skipping final sync.")
    print("Tip: Run mount_drive() and re-run this cell to back up manually.")

print("\nNotebook 03 complete.")
